In [1]:
import os, sys;
if not 'curdir' in globals(): curdir = os.getcwd(); os.chdir('..'); sys.path.append(os.getcwd());
import numpy as np
from src.spatial_audio_synthesizer import SpAudSyn

### Configuration 

In [2]:
# Same as data folder of DCASE 2026 Task 4
foreground_dir = 'data/dev_set/sound_event/train'
background_dir = 'data/dev_set/noise/train'
interference_dir = 'data/dev_set/interference/train'
sofa_dir = 'data/dev_set/room_ir/train'

config = {
    'duration': 10,  # seconds
    'sr': 32000,
    'max_event_overlap': 10,
    'max_event_dur': 10,
    'ref_db': -55,
    'foreground_dir': foreground_dir,
    'background_dir': background_dir,
    'interference_dir': interference_dir,
    'room_config': {
        'module': 'src.room',
        'main': 'SofaRoom',
        'args': {
            "path": sofa_dir,
            "direct_range_ms": [6, 50]
        }
    },
    'verbose': True,
}


### Synthesis

In [3]:
# Initialize object, room has also been initialized
spAudSynObj =  SpAudSyn(**config)

# Add sound events
for _ in range(2):
    spAudSynObj.add_event(
        label={'method': 'choose_wo_replacement'},
        source_file={'method': 'choose_wo_replacement', 'exclusion_folder_depth': 1},
        source_time={'method': 'choose'},
        event_time={'method': 'choose'},
        event_position={'method': 'choose', 'get_position_args': {'mode': 'point'}},
        snr={'method': 'uniform', 'range': [5, 30]},
        max_try=5,
        trim_amplitude=None,
        min_event_duration=None
    )

# Add interference events
spAudSynObj.add_interference(
        label={'method': 'choose_wo_replacement'},
        source_file={'method': 'choose_wo_replacement', 'exclusion_folder_depth': 1},
        source_time={'method': 'choose'},
        event_time={'method': 'choose'},
        event_position={'method': 'choose', 'get_position_args': {'mode': 'point'}},
        snr={'method': 'uniform', 'range': [5, 30]},
        max_try=5,
        trim_amplitude=None,
        min_event_duration=None
    )

# Add background noise
spAudSynObj.add_background(source_file={'method': 'choose'},);

# synthesize
soundscape = spAudSynObj.synthesize(fg_return={'wet': True,
                                              'dry': True,
                                              'rir': True,
                                              'metadata': True}, 
                                  int_return={'wet': True,
                                              'metadata': True},
                                   bg_return={'waveform': True,
                                              'metadata': True})

'''
soundscape = {
    'mixture': mixture
    'fg_events': [list of foreground events]
    'int_events': [list of interference events]
    'bg_events': [list of background noises]
}
# entry of each event (fg_events, int_events, bg_events)
entry = {
    'waveform'    : wet source
                    availabel if return_option['wet']==True
    'waveform_dry': dry source
                    availabel if return_option['dry']==True
    'rir'         : room impulse response
                    availabel if return_option['rir']==True
    'rir_dry'     : direct-path and early reflection RIR to calculate waveform_dry
                    availabel if return_option['rir']==True and return_option['dry']==True
    'metadata'    : the info entry of the event in SpatialSoundScene when calling add_event, add_interference, add_background, availabel if return_option['metadata']==True    
                    metadata containing: 'label', 'source_file', 'source_time', 'event_time', 'event_duration', 'event_position', 'snr', 'role'
}
'''
soundscape


Foreground labels: AlarmClock, Clapping, BicycleBell, Dishes, Percussion, VacuumCleaner, Cough, Doorbell, Blender, FootSteps, HairDryer, Pour, MechanicalFans, Buzzer, CupboardOpenClose, MusicalKeyboard, Speech, Typing


{'mixture': array([[ 5.70112839e-03,  5.59305027e-03,  5.57954051e-03, ...,
          1.73566449e-03,  1.89975880e-03,  1.88441360e-03],
        [-1.76978158e-03, -1.78329146e-03, -1.83733052e-03, ...,
          3.40992399e-06,  8.43692305e-05,  1.10104931e-04],
        [-1.56713487e-03, -1.56713487e-03, -1.58064463e-03, ...,
          8.23023607e-04,  8.67336414e-04,  8.03903440e-04],
        [-1.41852722e-03, -1.51309581e-03, -1.47256639e-03, ...,
          3.68275553e-04,  3.41738538e-04,  4.35055607e-04]]),
 'fg_events': [{'metadata': {'label': 'Dishes',
    'source_file': 'Dishes/50117.wav',
    'source_time': 0,
    'event_time': 0.17650344712591295,
    'event_duration': 1.1953125,
    'event_position': [[-0.12238193337490111,
      -0.6940624337987426,
      0.25651510749425155]],
    'snr': 14.044739145550796,
    'role': 'foreground'},
   'waveform': array([[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
       

### Store sound scene's metadata

In [4]:
spAudSynObj.generate_metadata('example/metadata.json');

### Create new object with metadata

In [5]:
spAudSynObj_load = SpAudSyn.from_metadata('example/metadata.json');
soundscape_load = spAudSynObj_load.synthesize(fg_return={'wet': True,
                                                         'dry': True,
                                                         'rir': True,
                                                         'metadata': True}, 
                                             int_return={'wet': True,
                                                         'metadata': True},
                                              bg_return={'waveform': True,
                                                         'metadata': True})

In [6]:
soundscape_load

{'mixture': array([[ 5.70112839e-03,  5.59305027e-03,  5.57954051e-03, ...,
          1.73566449e-03,  1.89975880e-03,  1.88441360e-03],
        [-1.76978158e-03, -1.78329146e-03, -1.83733052e-03, ...,
          3.40992399e-06,  8.43692305e-05,  1.10104931e-04],
        [-1.56713487e-03, -1.56713487e-03, -1.58064463e-03, ...,
          8.23023607e-04,  8.67336414e-04,  8.03903440e-04],
        [-1.41852722e-03, -1.51309581e-03, -1.47256639e-03, ...,
          3.68275553e-04,  3.41738538e-04,  4.35055607e-04]]),
 'fg_events': [{'metadata': {'label': 'Dishes',
    'source_file': 'Dishes/50117.wav',
    'source_time': 0,
    'event_time': 0.17650344712591295,
    'event_duration': 1.1953125,
    'event_position': [[-0.12238193337490111,
      -0.6940624337987426,
      0.25651510749425155]],
    'snr': 14.044739145550796,
    'role': 'foreground'},
   'waveform': array([[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
       